In [1]:
import pandas as pd
import numpy as np


In [2]:
messages = pd.read_csv("taskB_messages_public.csv")
candidates = pd.read_csv("taskB_candidates_public.csv")

print("Messages shape:", messages.shape)
print("Candidates shape:", candidates.shape)

Messages shape: (581, 5)
Candidates shape: (434, 11)


In [3]:
print("TASK B MESSAGES")
print(messages.head())

print("\nTASK B CANDIDATES")
print(candidates.head())

TASK B MESSAGES
  episode_id message_id          sender         timestamp  \
0       C001  C001-M001  Keeper_Absalom  2026-03-18 06:40   
1       C001  C001-M002      Rurik_Osei  2026-03-18 07:34   
2       C001  C001-M003      Rurik_Osei  2026-03-18 08:09   
3       C001  C001-M004     Ludmila_Vry  2026-03-18 08:20   
4       C001  C001-M005     Ludmila_Vry  2026-03-18 09:22   

                                                text  
0  The fog signal cut out twice overnight, both t...  
1  The helipad markings have faded to almost noth...  
2  Then I'll watch the forecast and do it the fir...  
3  Compressor or the timer relay? It did the same...  
4  Stores delivery has been pushed to the 29th be...  

TASK B CANDIDATES
  episode_id    gap_id  gap_position candidate_id  \
0       C001  C001-G01             3  C001-G01-C3   
1       C001  C001-G01             3  C001-G01-C2   
2       C001  C001-G01             3  C001-G01-C4   
3       C001  C001-G01             3  C001-G01-C1   
4  

In [4]:
print("\nMessages columns:")
print(messages.columns.tolist())

print("\nCandidates columns:")
print(candidates.columns.tolist())


Messages columns:
['episode_id', 'message_id', 'sender', 'timestamp', 'text']

Candidates columns:
['episode_id', 'gap_id', 'gap_position', 'candidate_id', 'candidate_text', 'candidate_sender', 'candidate_timestamp', 'feature_sender_pattern_fit', 'feature_timestamp_gap_fit', 'feature_thread_position_fit', 'feature_embedding_similarity']


In [5]:
gap_counts = candidates.groupby("gap_id").size()

print("Number of gaps:", len(gap_counts))
print("\nCandidates per gap:")
print(gap_counts.value_counts().sort_index())

Number of gaps: 88

Candidates per gap:
4    15
5    64
6     9
Name: count, dtype: int64


In [9]:
# STEP 1: Calculate final score

candidates["final_score"] = (
    0.20 * candidates["feature_sender_pattern_fit"]
    + 0.15 * candidates["feature_timestamp_gap_fit"]
    + 0.30 * candidates["feature_thread_position_fit"]
    + 0.35 * candidates["feature_embedding_similarity"]
)


# STEP 2: Sort candidates within each gap
# Highest score should come first

candidates = candidates.sort_values(
    by=["gap_id", "final_score"],
    ascending=[True, False]
).copy()


# STEP 3: Create rank for each gap

candidates["rank"] = (
    candidates.groupby("gap_id").cumcount() + 1
)


# STEP 4: Display the result

print(
    candidates[
        ["gap_id", "candidate_id", "final_score", "rank"]
    ].head(30)
)

      gap_id candidate_id  final_score  rank
0   C001-G01  C001-G01-C3      0.70870     1
1   C001-G01  C001-G01-C2      0.32500     2
2   C001-G01  C001-G01-C4      0.13635     3
3   C001-G01  C001-G01-C1      0.12085     4
4   C002-G01  C002-G01-C4      0.84885     1
5   C002-G01  C002-G01-C1      0.15600     2
6   C002-G01  C002-G01-C2      0.14450     3
7   C002-G01  C002-G01-C3      0.11890     4
8   C003-G01  C003-G01-C2      0.82140     1
9   C003-G01  C003-G01-C4      0.14955     2
10  C003-G01  C003-G01-C3      0.14715     3
11  C003-G01  C003-G01-C1      0.12310     4
12  C004-G01  C004-G01-C5      0.80595     1
13  C004-G01  C004-G01-C3      0.66690     2
15  C004-G01  C004-G01-C1      0.14270     3
14  C004-G01  C004-G01-C2      0.13605     4
16  C004-G01  C004-G01-C4      0.11805     5
18  C005-G01  C005-G01-C2      0.72150     1
17  C005-G01  C005-G01-C1      0.59320     2
19  C005-G01  C005-G01-C3      0.37820     3
20  C005-G01  C005-G01-C5      0.15745     4
21  C005-G

In [10]:
print(
    candidates[
        ["gap_id", "candidate_id", "final_score", "rank"]
    ].head(30)
)

      gap_id candidate_id  final_score  rank
0   C001-G01  C001-G01-C3      0.70870     1
1   C001-G01  C001-G01-C2      0.32500     2
2   C001-G01  C001-G01-C4      0.13635     3
3   C001-G01  C001-G01-C1      0.12085     4
4   C002-G01  C002-G01-C4      0.84885     1
5   C002-G01  C002-G01-C1      0.15600     2
6   C002-G01  C002-G01-C2      0.14450     3
7   C002-G01  C002-G01-C3      0.11890     4
8   C003-G01  C003-G01-C2      0.82140     1
9   C003-G01  C003-G01-C4      0.14955     2
10  C003-G01  C003-G01-C3      0.14715     3
11  C003-G01  C003-G01-C1      0.12310     4
12  C004-G01  C004-G01-C5      0.80595     1
13  C004-G01  C004-G01-C3      0.66690     2
15  C004-G01  C004-G01-C1      0.14270     3
14  C004-G01  C004-G01-C2      0.13605     4
16  C004-G01  C004-G01-C4      0.11805     5
18  C005-G01  C005-G01-C2      0.72150     1
17  C005-G01  C005-G01-C1      0.59320     2
19  C005-G01  C005-G01-C3      0.37820     3
20  C005-G01  C005-G01-C5      0.15745     4
21  C005-G

In [11]:
submission = candidates[
    ["gap_id", "candidate_id", "rank"]
].copy()

submission.to_csv(
    "submission_taskB.csv",
    index=False
)

print("submission_taskB.csv created successfully!")
print(submission.head(20))

submission_taskB.csv created successfully!
      gap_id candidate_id  rank
0   C001-G01  C001-G01-C3     1
1   C001-G01  C001-G01-C2     2
2   C001-G01  C001-G01-C4     3
3   C001-G01  C001-G01-C1     4
4   C002-G01  C002-G01-C4     1
5   C002-G01  C002-G01-C1     2
6   C002-G01  C002-G01-C2     3
7   C002-G01  C002-G01-C3     4
8   C003-G01  C003-G01-C2     1
9   C003-G01  C003-G01-C4     2
10  C003-G01  C003-G01-C3     3
11  C003-G01  C003-G01-C1     4
12  C004-G01  C004-G01-C5     1
13  C004-G01  C004-G01-C3     2
15  C004-G01  C004-G01-C1     3
14  C004-G01  C004-G01-C2     4
16  C004-G01  C004-G01-C4     5
18  C005-G01  C005-G01-C2     1
17  C005-G01  C005-G01-C1     2
19  C005-G01  C005-G01-C3     3


In [12]:
print("Checking submission...")

# Check duplicate rows
print("Duplicate rows:",
      submission.duplicated().sum())

# Check missing values
print("Missing values:")
print(submission.isnull().sum())

Checking submission...
Duplicate rows: 0
Missing values:
gap_id          0
candidate_id    0
rank            0
dtype: int64


In [13]:
valid = True

for gap_id, group in submission.groupby("gap_id"):

    expected = list(range(1, len(group) + 1))
    actual = sorted(group["rank"].tolist())

    if actual != expected:
        print("Problem in:", gap_id)
        print("Expected:", expected)
        print("Actual:", actual)
        valid = False

if valid:
    print("All gap rankings are valid!")
else:
    print("Some rankings need correction.")

All gap rankings are valid!


In [16]:
files.download("submission_taskB.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:
print("Messages:", messages.shape)
print("Candidates:", candidates.shape)

print("\nNumber of gaps:")
print(candidates["gap_id"].nunique())

print("\nCandidates per gap:")
print(candidates.groupby("gap_id").size().value_counts().sort_index())

print("\nFeature statistics:")
print(
    candidates[
        [
            "feature_sender_pattern_fit",
            "feature_timestamp_gap_fit",
            "feature_thread_position_fit",
            "feature_embedding_similarity"
        ]
    ].describe()
)

Messages: (581, 5)
Candidates: (434, 13)

Number of gaps:
88

Candidates per gap:
4    15
5    64
6     9
Name: count, dtype: int64

Feature statistics:
       feature_sender_pattern_fit  feature_timestamp_gap_fit  \
count                  434.000000                 434.000000   
mean                     0.425177                   0.400770   
std                      0.348464                   0.287048   
min                      0.020000                   0.108000   
25%                      0.091000                   0.186250   
50%                      0.260500                   0.264000   
75%                      0.792750                   0.780500   
max                      0.980000                   0.980000   

       feature_thread_position_fit  feature_embedding_similarity  
count                   434.000000                    434.000000  
mean                      0.375417                      0.455578  
std                       0.297311                      0.280449  
mi

In [17]:
feature_cols = [
    "feature_sender_pattern_fit",
    "feature_timestamp_gap_fit",
    "feature_thread_position_fit",
    "feature_embedding_similarity"
]

correlation = candidates[feature_cols].corr()

print("Correlation between Task B features:")
print(correlation.round(3))

Correlation between Task B features:
                              feature_sender_pattern_fit  \
feature_sender_pattern_fit                         1.000   
feature_timestamp_gap_fit                          0.518   
feature_thread_position_fit                        0.591   
feature_embedding_similarity                       0.688   

                              feature_timestamp_gap_fit  \
feature_sender_pattern_fit                        0.518   
feature_timestamp_gap_fit                         1.000   
feature_thread_position_fit                       0.473   
feature_embedding_similarity                      0.581   

                              feature_thread_position_fit  \
feature_sender_pattern_fit                          0.591   
feature_timestamp_gap_fit                           0.473   
feature_thread_position_fit                         1.000   
feature_embedding_similarity                        0.604   

                              feature_embedding_similarity  

In [18]:
for col in feature_cols:
    print("\n", col)
    print(candidates.groupby("gap_id")[col].agg(["min", "max", "mean"]).head())


 feature_sender_pattern_fit
            min    max     mean
gap_id                         
C001-G01  0.044  0.882  0.28375
C002-G01  0.044  0.900  0.27275
C003-G01  0.050  0.894  0.27300
C004-G01  0.020  0.914  0.38260
C005-G01  0.076  0.854  0.40540

 feature_timestamp_gap_fit
            min    max     mean
gap_id                         
C001-G01  0.174  0.804  0.34850
C002-G01  0.131  0.828  0.32875
C003-G01  0.134  0.800  0.33775
C004-G01  0.154  0.888  0.44660
C005-G01  0.165  0.856  0.33520

 feature_thread_position_fit
            min    max     mean
gap_id                         
C001-G01  0.096  0.873  0.31625
C002-G01  0.125  0.853  0.31575
C003-G01  0.090  0.818  0.29175
C004-G01  0.078  0.817  0.25660
C005-G01  0.086  0.763  0.36340

 feature_embedding_similarity
            min    max     mean
gap_id                         
C001-G01  0.113  0.888  0.33950
C002-G01  0.117  0.825  0.33850
C003-G01  0.146  0.792  0.33575
C004-G01  0.169  0.844  0.43840
C005-G01  0.135  0

In [19]:
# ============================================
# IMPROVED TASK B RANKING
# Normalize features within each gap
# ============================================

feature_cols = [
    "feature_sender_pattern_fit",
    "feature_timestamp_gap_fit",
    "feature_thread_position_fit",
    "feature_embedding_similarity"
]

# Create a copy so the previous result is not disturbed
improved = candidates.copy()

# Normalize each feature within every gap
for col in feature_cols:
    gap_min = improved.groupby("gap_id")[col].transform("min")
    gap_max = improved.groupby("gap_id")[col].transform("max")

    # Min-max normalization
    improved[col + "_norm"] = (
        (improved[col] - gap_min) /
        (gap_max - gap_min)
    )

    # If all candidates have the same value,
    # give them equal normalized score
    improved[col + "_norm"] = improved[col + "_norm"].fillna(0.5)


# Combine normalized features
improved["improved_score"] = (
    0.20 * improved["feature_sender_pattern_fit_norm"]
    + 0.15 * improved["feature_timestamp_gap_fit_norm"]
    + 0.30 * improved["feature_thread_position_fit_norm"]
    + 0.35 * improved["feature_embedding_similarity_norm"]
)


# Sort candidates inside each gap
improved = improved.sort_values(
    by=["gap_id", "improved_score"],
    ascending=[True, False]
).copy()


# Assign rank
improved["improved_rank"] = (
    improved.groupby("gap_id").cumcount() + 1
)


# Display results
print(
    improved[
        [
            "gap_id",
            "candidate_id",
            "improved_score",
            "improved_rank"
        ]
    ].head(30)
)

      gap_id candidate_id  improved_score  improved_rank
0   C001-G01  C001-G01-C3        0.807876              1
1   C001-G01  C001-G01-C2        0.272162              2
2   C001-G01  C001-G01-C4        0.043344              3
3   C001-G01  C001-G01-C1        0.021002              4
4   C002-G01  C002-G01-C4        1.000000              1
5   C002-G01  C002-G01-C1        0.066892              2
6   C002-G01  C002-G01-C2        0.052323              3
7   C002-G01  C002-G01-C3        0.017217              4
8   C003-G01  C003-G01-C2        1.000000              1
9   C003-G01  C003-G01-C4        0.061527              2
10  C003-G01  C003-G01-C3        0.056868              3
11  C003-G01  C003-G01-C1        0.020316              4
12  C004-G01  C004-G01-C5        0.931934              1
13  C004-G01  C004-G01-C3        0.743437              2
15  C004-G01  C004-G01-C1        0.046813              3
14  C004-G01  C004-G01-C2        0.031404              4
16  C004-G01  C004-G01-C4      

In [20]:
comparison = candidates[
    ["gap_id", "candidate_id", "rank"]
].merge(
    improved[
        ["gap_id", "candidate_id", "improved_rank"]
    ],
    on=["gap_id", "candidate_id"]
)

print(comparison.head(30))

      gap_id candidate_id  rank  improved_rank
0   C001-G01  C001-G01-C3     1              1
1   C001-G01  C001-G01-C2     2              2
2   C001-G01  C001-G01-C4     3              3
3   C001-G01  C001-G01-C1     4              4
4   C002-G01  C002-G01-C4     1              1
5   C002-G01  C002-G01-C1     2              2
6   C002-G01  C002-G01-C2     3              3
7   C002-G01  C002-G01-C3     4              4
8   C003-G01  C003-G01-C2     1              1
9   C003-G01  C003-G01-C4     2              2
10  C003-G01  C003-G01-C3     3              3
11  C003-G01  C003-G01-C1     4              4
12  C004-G01  C004-G01-C5     1              1
13  C004-G01  C004-G01-C3     2              2
14  C004-G01  C004-G01-C1     3              3
15  C004-G01  C004-G01-C2     4              4
16  C004-G01  C004-G01-C4     5              5
17  C005-G01  C005-G01-C2     1              1
18  C005-G01  C005-G01-C1     2              2
19  C005-G01  C005-G01-C3     3              3
20  C005-G01 

In [21]:
# Select the first gap
gap_id = candidates["gap_id"].iloc[0]

gap_info = candidates[candidates["gap_id"] == gap_id].copy()

print("Gap ID:", gap_id)
print("\nCandidate messages:")
print(
    gap_info[
        [
            "candidate_id",
            "candidate_text",
            "candidate_sender",
            "candidate_timestamp",
            "feature_sender_pattern_fit",
            "feature_timestamp_gap_fit",
            "feature_thread_position_fit",
            "feature_embedding_similarity"
        ]
    ].to_string(index=False)
)

Gap ID: C001-G01

Candidate messages:
candidate_id                                                                 candidate_text candidate_sender candidate_timestamp  feature_sender_pattern_fit  feature_timestamp_gap_fit  feature_thread_position_fit  feature_embedding_similarity
 C001-G01-C3 That's a repaint in dry weather, which gives us about three days a month here.      Ludmila_Vry    2026-03-18 07:54                       0.077                      0.804                        0.873                         0.888
 C001-G01-C2                The gate intercom crackles so badly nobody can hear the caller.   Silas_Obiefune    2026-03-18 08:01                       0.882                      0.227                        0.166                         0.185
 C001-G01-C4       The tea fund is in deficit, which is a sentence I never wanted to write.   Hadrian_Vosloo    2026-03-18 07:48                       0.044                      0.189                        0.130                     

In [22]:
episode_id = gap_info["episode_id"].iloc[0]

print("Episode:", episode_id)

episode_messages = messages[
    messages["episode_id"] == episode_id
].copy()

episode_messages["timestamp"] = pd.to_datetime(
    episode_messages["timestamp"]
)

episode_messages = episode_messages.sort_values("timestamp")

print("\nMessages in this episode:")
print(
    episode_messages[
        ["message_id", "sender", "timestamp", "text"]
    ].to_string(index=False)
)

Episode: C001

Messages in this episode:
message_id         sender           timestamp                                                                                text
 C001-M001 Keeper_Absalom 2026-03-18 06:40:00          The fog signal cut out twice overnight, both times for about four minutes.
 C001-M002     Rurik_Osei 2026-03-18 07:34:00              The helipad markings have faded to almost nothing on the eastern half.
 C001-M003     Rurik_Osei 2026-03-18 08:09:00                Then I'll watch the forecast and do it the first dry morning we get.
 C001-M004    Ludmila_Vry 2026-03-18 08:20:00  Compressor or the timer relay? It did the same in the autumn and it was the relay.
 C001-M005    Ludmila_Vry 2026-03-18 09:22:00                   Stores delivery has been pushed to the 29th because of the swell.
 C001-M006 Keeper_Absalom 2026-03-18 09:24:00                            We're fine for food but low on diesel for the generator.
 C001-M007     Rurik_Osei 2026-03-18 10:15:00    

In [23]:
gap_position = gap_info["gap_position"].iloc[0]

print("Gap position:", gap_position)
print("Episode:", episode_id)

print("\nCandidates:")
print(
    gap_info[
        ["candidate_id", "candidate_sender", "candidate_timestamp", "candidate_text"]
    ].to_string(index=False)
)

Gap position: 3
Episode: C001

Candidates:
candidate_id candidate_sender candidate_timestamp                                                                 candidate_text
 C001-G01-C3      Ludmila_Vry    2026-03-18 07:54 That's a repaint in dry weather, which gives us about three days a month here.
 C001-G01-C2   Silas_Obiefune    2026-03-18 08:01                The gate intercom crackles so badly nobody can hear the caller.
 C001-G01-C4   Hadrian_Vosloo    2026-03-18 07:48       The tea fund is in deficit, which is a sentence I never wanted to write.
 C001-G01-C1   Isolda_Nwamaka    2026-03-18 07:41               The signage for the queue system contradicts what the staff say.


In [24]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd

In [26]:
def get_context_score(candidate_text, episode_texts):
    """
    Measures how similar a candidate message is
    to the surrounding conversation.
    """

    texts = [candidate_text] + episode_texts

    vectorizer = TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        ngram_range=(1, 2)
    )

    tfidf_matrix = vectorizer.fit_transform(texts)

    candidate_vector = tfidf_matrix[0]
    context_vectors = tfidf_matrix[1:]

    similarities = cosine_similarity(
        candidate_vector,
        context_vectors
    )[0]

    if len(similarities) == 0:
        return 0.0

    return float(np.mean(similarities))

In [27]:
context_scores = []

for idx, row in candidates.iterrows():

    episode_id = row["episode_id"]

    candidate_time = pd.to_datetime(
        row["candidate_timestamp"]
    )

    # Get messages from the same episode
    episode = messages[
        messages["episode_id"] == episode_id
    ].copy()

    episode["timestamp"] = pd.to_datetime(
        episode["timestamp"]
    )

    episode = episode.sort_values("timestamp")

    # Find messages before candidate time
    before = episode[
        episode["timestamp"] < candidate_time
    ]

    # Find messages after candidate time
    after = episode[
        episode["timestamp"] > candidate_time
    ]

    # Take nearby messages
    before_texts = before["text"].tail(2).tolist()
    after_texts = after["text"].head(2).tolist()

    surrounding_texts = before_texts + after_texts

    # Calculate context similarity
    score = get_context_score(
        row["candidate_text"],
        surrounding_texts
    )

    context_scores.append(score)


candidates["context_similarity"] = context_scores

In [28]:
print(
    candidates[
        [
            "gap_id",
            "candidate_id",
            "context_similarity"
        ]
    ].head(30)
)

      gap_id candidate_id  context_similarity
0   C001-G01  C001-G01-C3            0.016953
1   C001-G01  C001-G01-C2            0.000000
2   C001-G01  C001-G01-C4            0.000000
3   C001-G01  C001-G01-C1            0.000000
4   C002-G01  C002-G01-C4            0.000000
5   C002-G01  C002-G01-C1            0.000000
6   C002-G01  C002-G01-C2            0.000000
7   C002-G01  C002-G01-C3            0.000000
8   C003-G01  C003-G01-C2            0.031492
9   C003-G01  C003-G01-C4            0.000000
10  C003-G01  C003-G01-C3            0.000000
11  C003-G01  C003-G01-C1            0.000000
12  C004-G01  C004-G01-C5            0.045427
13  C004-G01  C004-G01-C3            0.015278
15  C004-G01  C004-G01-C1            0.000000
14  C004-G01  C004-G01-C2            0.013044
16  C004-G01  C004-G01-C4            0.000000
18  C005-G01  C005-G01-C2            0.054167
17  C005-G01  C005-G01-C1            0.045565
19  C005-G01  C005-G01-C3            0.000000
20  C005-G01  C005-G01-C5         

In [29]:
candidates["context_score_final"] = (
    0.15 * candidates["feature_sender_pattern_fit"]
    + 0.10 * candidates["feature_timestamp_gap_fit"]
    + 0.25 * candidates["feature_thread_position_fit"]
    + 0.30 * candidates["feature_embedding_similarity"]
    + 0.20 * candidates["context_similarity"]
)


In [30]:
improved_context = candidates.sort_values(
    by=["gap_id", "context_score_final"],
    ascending=[True, False]
).copy()

improved_context["context_rank"] = (
    improved_context.groupby("gap_id").cumcount() + 1
)

In [31]:
print(
    improved_context[
        improved_context["gap_id"] == "C001-G01"
    ][
        [
            "candidate_id",
            "candidate_text",
            "feature_sender_pattern_fit",
            "feature_timestamp_gap_fit",
            "feature_thread_position_fit",
            "feature_embedding_similarity",
            "context_similarity",
            "context_score_final",
            "context_rank"
        ]
    ].to_string(index=False)
)

candidate_id                                                                 candidate_text  feature_sender_pattern_fit  feature_timestamp_gap_fit  feature_thread_position_fit  feature_embedding_similarity  context_similarity  context_score_final  context_rank
 C001-G01-C3 That's a repaint in dry weather, which gives us about three days a month here.                       0.077                      0.804                        0.873                         0.888            0.016953             0.579991             1
 C001-G01-C2                The gate intercom crackles so badly nobody can hear the caller.                       0.882                      0.227                        0.166                         0.185            0.000000             0.252000             2
 C001-G01-C4       The tea fund is in deficit, which is a sentence I never wanted to write.                       0.044                      0.189                        0.130                         0.172            